In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, Flatten, GlobalAveragePooling2D, Concatenate, Dropout
from tensorflow.keras.applications import InceptionV3
from sklearn.model_selection import train_test_split

# Load handcrafted features
data = pd.read_csv("../Dataset/Features/signature_features.csv")
labels = data["label"].values
handcrafted_features = data.iloc[:, 2:].values  # Skip filename and label columns

# Image settings
image_size = (299, 299)  # InceptionV3 default input size
batch_size = 32  

# Generate image paths
image_paths = [f"../Dataset/Processing/AugmentedDataset/{'Genuine' if label == 0 else 'Forged'}/{file}" 
               for file, label in zip(data["filename"], labels)]

# Split dataset
X_train_img, X_test_img, X_train_handcrafted, X_test_handcrafted, y_train, y_test = train_test_split(
    image_paths, handcrafted_features, labels, test_size=0.2, random_state=42, stratify=labels)


# Custom Data Generator for Lazy Loading
class SignatureDataGenerator(Sequence):
    def __init__(self, image_paths, handcrafted_features, labels, batch_size, image_size, shuffle=True):
        self.image_paths = image_paths
        self.handcrafted_features = handcrafted_features
        self.labels = labels
        self.batch_size = batch_size
        self.image_size = image_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.image_paths))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return int(np.floor(len(self.image_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch_image_paths = [self.image_paths[i] for i in batch_indexes]
        batch_handcrafted = self.handcrafted_features[batch_indexes]
        batch_labels = self.labels[batch_indexes]

        # Load images on the fly
        images = np.array([
            tf.keras.preprocessing.image.img_to_array(
                tf.keras.preprocessing.image.load_img(path, target_size=self.image_size)
            ) for path in batch_image_paths
        ])
        images = tf.keras.applications.inception_v3.preprocess_input(images)

        # Reshape handcrafted features for LSTM
        batch_handcrafted = batch_handcrafted.reshape(-1, batch_handcrafted.shape[1], 1)

        return [images, batch_handcrafted], batch_labels

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)


# Create generators for training and validation
train_generator = SignatureDataGenerator(X_train_img, X_train_handcrafted, y_train, batch_size, image_size)
val_generator = SignatureDataGenerator(X_test_img, X_test_handcrafted, y_test, batch_size, image_size, shuffle=False)


# CNN Feature Extractor (InceptionV3)
inception_base = InceptionV3(weights=None, include_top=False, input_shape=(299, 299, 3))
cnn_output = GlobalAveragePooling2D()(inception_base.output)

# LSTM for sequence modeling
lstm_input = Input(shape=(X_train_handcrafted.shape[1], 1))  # Handcrafted features
lstm_layer = LSTM(128, return_sequences=False)(lstm_input)

# Combine CNN and LSTM
merged = Concatenate()([cnn_output, lstm_layer])
dense_layer = Dense(128, activation='relu')(merged)
dropout_layer = Dropout(0.5)(dense_layer)
output_layer = Dense(1, activation='sigmoid')(dropout_layer)

# Define hybrid model
model = Model(inputs=[inception_base.input, lstm_input], outputs=output_layer)

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model using data generators
model.fit(train_generator, validation_data=val_generator, epochs=25)

# Save model
model.save("../Model/signature_forgery_detection_model.h5")
print("Model training completed and saved!")

# Load the trained model
model = load_model("../Model/signature_forgery_detection_model.h5")
